# Data Analysis and thedress dataset

*In 2015 a scandal has happened on twitter regarding a published picture showing a dress, lot of people has argued regarding its color, some people saw it black and blue, others saw it white and gold. Actually the dress was black and blue, and the dress has teasing the attention of scientists to understand why people saw it in different colors.*

This dataset contains votes of people from different countries who has indicated in which color do they see the dress ?


## Dataset's features descriptions

**_unit_id**: id of the vote ? <br/>
**_created_at**: creation date of the vote<br/>
**_id**: another id ?<br/>
**_started_at**: date<br/>
**_tainted**: ?<br/>
**_channel**: source of the vote<br/>
**_trust**: probability of trust on the voter whether has told the truth regarding the color<br/>
**_worker_id**: id of the voter ?<br/>
**_country**: origin country of the voter<br/>
**_region**: geographical region of the voter<br/>
**_city**: city the voter is comming from<br/>
**_ip**: voter's ip adress<br/>
**_color1**: first color of the dress<br/>
**_color2**: second color of the dress<br/>
**_you**: the assigned sentence given to the voter regarding his vote<br/>
**_proccessed_color_combo**: choosen colors<br/>
**what group ?**: assigned group of the voter<br/>

In this notebook, we will study the data and its relevant features regarding:
- the country of the voter
- the dominant vote
- the link between the assigned trust with other feature

*set the seed*

In [ ]:
import random

random.seed(10)

In [ ]:
import pandas
import numpy
df = pandas.read_csv("/kaggle/input/the-colors-of-the-iconic-dress/Hashtag-That-Dress-DFE.csv", encoding='latin-1')
df.columns

In [ ]:
df["_tainted"].unique()

In [ ]:
df["_channel"].unique()

In [ ]:
print(f'{df["_trust"].std()}, {df["_trust"].mean()}, {df["_trust"].min()}, {df["_trust"].max()}')

In [ ]:
df["_worker_id"].duplicated().all() # no doublons

In [ ]:
df["_country"].unique()

In [ ]:
countries_col = df["_country"].copy()

In [ ]:
countries_df = df[["_worker_id", "_country"]].groupby("_country").count().sort_values(by="_worker_id", ascending=False).rename(columns={'_worker_id':'votes'})
countries_df["country"] = countries_df.index
countries_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn

In [ ]:
plt.xticks(rotation=90)
seaborn.barplot(data=countries_df[:20], x="country", y="votes").set(title='Bar plot of the most occured country voting to thedress survey')


# TOP 6 countries vs rest of the world votes proportions

In [ ]:
# build condensed dataset per countries
rest = countries_df.index[1]

rest_count = countries_df["votes"][6:].sum()
rest_count

countries_labels = countries_df.index[:6].to_list() + ["others"]
countries_val = countries_df["votes"][:6].to_list() + [rest_count]

condensed_countries_df = pandas.DataFrame({
    "countries_and_rest": countries_labels,
    "votes": countries_val
})
condensed_countries_df

In [ ]:
plt.pie(condensed_countries_df["votes"], labels=condensed_countries_df["countries_and_rest"], labeldistance=1.15, wedgeprops = { 'linewidth' : 3, 'edgecolor' : 'white' });

# Vote proportion

In [ ]:
df[df["processed_color_combo"] == "white & gold"].shape

In [ ]:
df[df["processed_color_combo"] == "black & blue"].shape

In [ ]:
other_proportion = len(df) - 373 - 470
other_proportion

In [ ]:
labels = ["white & gold", "black & blue", "other"]
sizes = [470, 373, 172]
plt.pie(sizes, labels=labels, labeldistance=1.15, wedgeprops = { 'linewidth' : 3, 'edgecolor' : 'white' });

# Does vote differs from countries ?

In [ ]:
countries_total_vote = df[["_worker_id", "_country"]].groupby(by="_country").count().reset_index().rename(columns={"_worker_id": "votes"})
countries_color_votes = df[["_worker_id", "_country", "processed_color_combo"]].groupby(by=["_country", "processed_color_combo"]).count().reset_index().rename(columns={"_worker_id": "votes"})

def get_total_vote_per_country(row, total_vote):
    return total_vote[total_vote["_country"] == row["_country"]].values[0][1]

countries_color_votes["total"] = countries_color_votes.apply(lambda row: get_total_vote_per_country(row, countries_total_vote), axis=1)

countries_color_votes["proportions"] = countries_color_votes["votes"] / countries_color_votes["total"]

countries_color_votes

In [ ]:
countries_color_votes["total"].mean()

In [ ]:
# take only countries that are above the mean 
countries_color_votes = countries_color_votes[countries_color_votes["total"] > countries_color_votes["total"].mean()]


In [ ]:

# plot blackAndBlue

blackAndBlue = countries_color_votes[countries_color_votes["processed_color_combo"] == "black & blue"].sort_values(by="proportions", ascending=False)

plt.xticks(rotation=90)
seaborn.barplot(data=blackAndBlue[:20], x="_country", y="proportions").set(title='Bar plot proportion per country on black & blue votes')

In [ ]:
blackAndBlue["proportions"].std()

In [ ]:
whiteAndGold = countries_color_votes[countries_color_votes["processed_color_combo"] == "white & gold"].sort_values(by="proportions", ascending=False)

plt.xticks(rotation=90)
seaborn.barplot(data=whiteAndGold[:20], x="_country", y="proportions").set(title='Bar plot proportion per country on black & blue votes')

In [ ]:
whiteAndGold["proportions"].std()

In [ ]:
len(countries_color_votes["_country"].unique())

# Trust

In [ ]:
blackAndblue = df[df["processed_color_combo"] == "black & blue"]
whiteAndgold = df[df["processed_color_combo"] == "white & gold"]
others = df[(df["processed_color_combo"] != "black & blue") & (df["processed_color_combo"] != "white & gold")]

*mean trust over the different type of votes*

In [ ]:
print(f'{blackAndblue["_trust"].mean()} , {whiteAndgold["_trust"].mean()}, {others["_trust"].mean()}')

# In which criteria put their trust ?

In [ ]:
mean = df["_trust"].mean()
mean

## channels / sources ?

In [ ]:
more_trusted_votes = df[df["_trust"] > mean]
more_trusted_votes[:10]

In [ ]:
trustAndChannel = more_trusted_votes[["_trust", "_channel"]].groupby(by="_channel").describe().sort_values(by=("_trust","count"), ascending=False)
trustAndChannel[:5]

## countries ?

In [ ]:
more_trusted_votes[["_trust", "_country"]].groupby(by="_country").describe().sort_values(by=("_trust", "mean"), ascending=False)[:10]

In [ ]:
more_trusted_votes[["_trust", "_country"]].groupby(by="_country").describe().sort_values(by=("_trust", "count"), ascending=False)

In [ ]:
countries_df[:10]

*doesn't seems to have any correlations with countries and their number of votes*

## Date ?

In [ ]:
df["_created_at"]

In [ ]:
df["_started_at"]

In [ ]:
# convert columns into date
df["_started_at"] = pandas.to_datetime(df["_started_at"])
df["_created_at"] = pandas.to_datetime(df["_created_at"])

In [ ]:
diff = (df['_started_at'] - df['_created_at']).dt.seconds
print(f'{diff.min()} {diff.max()} {diff.mean()}')

In [ ]:
# add diff date
df.insert(0, "diff date", diff)

**trust and date difference seems not super correlated**

In [ ]:
df[["_trust", "diff date"]].corr()

In [ ]:

x = numpy.arange(len(df))

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.plot(x, df["_trust"])
ax2.scatter(x, df["diff date"])

fig.show()